# Đọc GPT-2 small từ bên trong: head attention, logit lens, probe

Bạn nạp **GPT-2 small** (124 triệu tham số, OpenAI 2019) vào chính lớp `GPT` đã viết ở module 05, rồi dùng công cụ của ba bài
Basic để đọc nó: tìm head previous-token, duplicate-token và induction, kiểm bằng ablation xem induction head có thật sự cần
cho việc chép lại chuỗi không, đọc residual stream qua từng tầng bằng logit lens, và dò bằng linear probe có đối chứng xem
thông tin "token này đã xuất hiện chưa" nằm ở tầng nào.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) tải model, chạy thí nghiệm; test bỏ qua chúng.
- Các cell thí nghiệm dùng `hooks.py`, `lens.py`, `probe.py` **của bạn** từ ba bài Basic (chưa xong thì dùng lời giải).
- CPU 4 luồng đủ cho cả notebook (vài phút); lần đầu tải GPT-2 từ Hugging Face (548 MB, vào cache `~/.cache/huggingface`).
- Chấm: `uv run pytest tracks/ai-engineer/modules/08-interpretability/exercises/intermediate/01-gpt2-heads-lens-probe -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, tải IWSLT15 (văn bản TED tiếng Anh) nếu chưa có
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
IWSLT = ROOT / "data/raw/iwslt15-en-vi"
if not (IWSLT.is_dir() and any(IWSLT.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("iwslt15-en-vi")
print(sorted(p.name for p in IWSLT.iterdir()))

In [ ]:
import contextlib

import numpy as np
import torch
import torch.nn.functional as F

## 1. GPT-2 trong lớp `GPT` của module 05

GPT-2 small là đúng kiến trúc bạn đã viết ở module 05: pre-norm, LayerNorm, positional embedding học được, MLP $$4d$$ với GELU,
weight tying, 12 tầng, 12 head, $$d = 768$$, context 1 024, vocab 50 257. Nạp trọng số của nó vào lớp `GPT` của bạn nghĩa là
mọi hook ở bài Basic 01 dùng được ngay, không cần thư viện interpretability. Hai chi tiết: GPT-2 dùng GELU xấp xỉ tanh, và lớp
`Conv1D` của nó lưu trọng số theo chiều ngược với `nn.Linear`.

In [ ]:
def gpt2_into_gpt_mini(gpt_module, hf_model):
    """Dựng `gpt_module.GPT` mang đúng trọng số của một `transformers.GPT2LMHeadModel` (GPT-2 gốc của OpenAI).

    - Config: `vocab_size`, `block_size` = `n_positions`, `n_layer`, `n_head`, `n_embd` lấy từ `hf_model.config`, dropout 0.
    - GPT-2 dùng GELU xấp xỉ tanh: thay `blocks[i].mlp.act` bằng `torch.nn.GELU(approximate="tanh")`.
    - Lớp `Conv1D` của GPT-2 lưu trọng số shape (in, out), ngược với `nn.Linear` (out, in).
    - Giữ weight tying (`lm_head.weight` là `tok_emb.weight`). Trả model ở chế độ eval, trên CPU.
    """
    c = hf_model.config
    model = gpt_module.GPT(gpt_module.GPTConfig(vocab_size=c.vocab_size, block_size=c.n_positions, n_layer=c.n_layer,
                                                n_head=c.n_head, n_embd=c.n_embd, dropout=0.0))
    for block in model.blocks:
        block.mlp.act = torch.nn.GELU(approximate="tanh")
    sd = {k: v.detach().cpu() for k, v in hf_model.state_dict().items()}
    new = {"tok_emb.weight": sd["transformer.wte.weight"], "pos_emb.weight": sd["transformer.wpe.weight"],
           "ln_f.weight": sd["transformer.ln_f.weight"], "ln_f.bias": sd["transformer.ln_f.bias"],
           "lm_head.weight": sd["transformer.wte.weight"]}
    names = {"ln_1": "ln1", "ln_2": "ln2", "attn.c_attn": "attn.qkv", "attn.c_proj": "attn.proj",
             "mlp.c_fc": "mlp.fc", "mlp.c_proj": "mlp.proj"}
    for i in range(c.n_layer):
        for hf_name, name in names.items():
            w = sd[f"transformer.h.{i}.{hf_name}.weight"]
            new[f"blocks.{i}.{name}.weight"] = w if hf_name.startswith("ln") else w.T.contiguous()
            new[f"blocks.{i}.{name}.bias"] = sd[f"transformer.h.{i}.{hf_name}.bias"]
    model.load_state_dict(new)
    return model.eval()

## 2. Ablation head và loss trong ngữ cảnh

Điểm attention chỉ là tương quan. Để biết một head có **cần** cho một hành vi không, tắt nó đi và đo hành vi. Hai cách tắt:
**zero ablation** (output head bằng 0) và **mean ablation** (thay bằng output trung bình của head đó trên văn bản thường). Hành
vi cần đo là chép lại chuỗi: trên dãy `[prefix] + r + r` với `r` ngẫu nhiên, bản thứ nhất không đoán được (loss cỡ $$\ln V$$),
bản thứ hai đoán được nếu model biết nhìn lại.

In [ ]:
@contextlib.contextmanager
def ablate_heads(model, heads, means=None):
    """Trong khối `with`, thay output của các head trong `heads` (list `(layer, head)`) trước khi vào `attn.proj`.

    `means is None`: zero ablation (output head bằng 0). Ngược lại `means[layer]` là tensor (n_head, head_dim) và output của
    head bị thay bằng `means[layer][head]` ở mọi vị trí (mean ablation). Head khác không đổi. Gỡ hook khi ra khỏi khối, kể cả
    khi có exception.
    """
    by_layer = {}
    for layer, head in heads:
        by_layer.setdefault(layer, []).append(head)
    n_head = model.config.n_head
    handles = []
    for layer, hs in by_layer.items():
        def hook(module, args, layer=layer, hs=hs):
            z = args[0].clone()
            B, T, C = z.shape
            z = z.view(B, T, n_head, C // n_head)
            for h in hs:
                z[:, :, h] = 0.0 if means is None else means[layer][h].to(z)
            return (z.view(B, T, C),)
        handles.append(model.blocks[layer].attn.proj.register_forward_pre_hook(hook))
    try:
        yield model
    finally:
        for handle in handles:
            handle.remove()

In [ ]:
def in_context_loss(model, seq, half):
    """`seq` là dãy `[prefix] + r + r` của `repeated_tokens` (nửa sau là `half` vị trí cuối). Trả `(loss_đầu, loss_sau)`
    kiểu float: cross-entropy trung bình (nat) khi dự đoán token thứ 2 tới thứ `half` của bản thứ nhất, và token thứ 2 tới
    thứ `half` của bản thứ hai. Token đầu của mỗi bản không đoán được nên bỏ. Không gradient.
    """
    T = seq.shape[1]
    start2 = T - half
    start1 = start2 - half
    with torch.no_grad():
        logits, _ = model(seq)
    nll = F.cross_entropy(logits[:, :-1].transpose(1, 2), seq[:, 1:], reduction="none")
    first = nll[:, start1:start1 + half - 1]
    second = nll[:, start2:start2 + half - 1]
    return first.mean().item(), second.mean().item()

## 3. Nhãn cho probe và thước đo của logit lens

- `repeated_token_labels`: nhãn của bài probe "token này đã xuất hiện trước đó trong ngữ cảnh chưa". Muốn biết nhãn này, model
  phải nhìn lại các vị trí trước: một embedding đứng một mình không biết được.
- `lens_metrics`: logit lens ở tầng giữa "gần" output thật tới đâu, đo bằng ba con số (loss với token thật, trùng top-1 với
  model, KL tới phân phối của model).

In [ ]:
def repeated_token_labels(idx):
    """idx (B, T) LongTensor -> mảng numpy int64 (B, T): 1 nếu token ở vị trí t đã xuất hiện ở một vị trí nhỏ hơn trong
    **cùng hàng** (không tính vị trí 0, thường là token đầu `<|endoftext|>`), ngược lại 0. Vị trí 0 luôn là 0.
    """
    idx = idx.cpu().numpy()
    out = np.zeros(idx.shape, dtype=np.int64)
    for b in range(idx.shape[0]):
        seen = set()
        for t in range(1, idx.shape[1]):
            v = int(idx[b, t])
            out[b, t] = v in seen
            seen.add(v)
    return out

In [ ]:
def lens_metrics(lens_logits, final_logits, targets):
    """So logits đọc bằng logit lens với logits thật của model, trên mọi vị trí. lens_logits, final_logits (..., V);
    targets (...). Trả dict float:
    - `ce`: cross-entropy trung bình của lens_logits với targets (nat);
    - `top1_agree`: tỉ lệ vị trí mà argmax của lens trùng argmax của model;
    - `kl`: KL(p_model || p_lens) trung bình theo vị trí (nat).
    """
    V = lens_logits.shape[-1]
    lens_logits, final_logits = lens_logits.reshape(-1, V).float(), final_logits.reshape(-1, V).float()
    targets = targets.reshape(-1)
    logp_lens = F.log_softmax(lens_logits, dim=-1)
    logp_model = F.log_softmax(final_logits, dim=-1)
    return {
        "ce": F.cross_entropy(lens_logits, targets).item(),
        "top1_agree": (lens_logits.argmax(-1) == final_logits.argmax(-1)).float().mean().item(),
        "kl": (logp_model.exp() * (logp_model - logp_lens)).sum(-1).mean().item(),
    }

## 4. Thí nghiệm

Các cell dưới không được chấm. Chúng nạp công cụ của bạn từ ba bài Basic, tải GPT-2, rồi chạy bốn thí nghiệm. Ghi lại các bảng
để làm trắc nghiệm và so với `SOLUTION.md`.

In [ ]:
# explore: nạp lớp GPT của module 05 và công cụ của ba bài Basic (bài nào chưa xong thì dùng lời giải)
import importlib.util
import time


def load_file(name, candidates, check):
    for path in candidates:
        spec = importlib.util.spec_from_file_location(name, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        spec.loader.exec_module(module)
        try:
            check(module)
            print("dùng", path.relative_to(ROOT))
            return module
        except NotImplementedError:
            continue


M05 = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"
M08 = ROOT / "tracks/ai-engineer/modules/08-interpretability"
tiny_idx = torch.zeros(1, 3, dtype=torch.long)
gpt_mini = load_file("gpt_mini", [M05 / "exercises/basic/01-gpt-mini/starter/gpt.py", M05 / "solutions/basic/01-gpt-mini/gpt.py"],
                     lambda m: m.GPT(m.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(tiny_idx))
tiny = gpt_mini.GPT(gpt_mini.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))
hooks = load_file("m08_hooks", [M08 / f"{d}/basic/01-hooks-residual-stream/{'starter/' if d == 'exercises' else ''}hooks.py" for d in ["exercises", "solutions"]],
                  lambda m: m.run_with_cache(tiny, tiny_idx))
lens = load_file("m08_lens", [M08 / f"{d}/basic/02-attention-logit-lens/{'starter/' if d == 'exercises' else ''}lens.py" for d in ["exercises", "solutions"]],
                 lambda m: m.prev_token_score(torch.ones(1, 1, 2, 2)))
probe = load_file("m08_probe", [M08 / f"{d}/basic/03-linear-probe-controls/{'starter/' if d == 'exercises' else ''}probe.py" for d in ["exercises", "solutions"]],
                  lambda m: m.majority_baseline(np.array([0]), np.array([0])))

In [ ]:
# explore: tải GPT-2 small (lần đầu khoảng 548 MB), nạp vào lớp GPT, kiểm khớp với bản Hugging Face
from transformers import AutoTokenizer, GPT2LMHeadModel

torch.set_num_threads(4)
hf = GPT2LMHeadModel.from_pretrained("openai-community/gpt2").eval()
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = gpt2_into_gpt_mini(gpt_mini, hf)
BOS = tok.bos_token_id
ids = torch.tensor([[BOS] + tok.encode("The Eiffel Tower is located in the city of")])
with torch.no_grad():
    diff = (hf(ids).logits - model(ids)[0]).abs().max().item()
print(f"{model.num_params():,} tham số | lệch logits lớn nhất so với Hugging Face: {diff:.1e}")
print("token kế tiếp:", tok.decode(model(ids)[0][0, -1].argmax()))
del hf

In [ ]:
# explore: văn bản TED (IWSLT15 tst2012 + tst2013, đã tách từ) -> văn bản thường, cắt thành ngữ cảnh BOS + 127 token
import html
import re


def detok(s):
    s = html.unescape(s)
    s = re.sub(r" ([.,!?;:%)\]])", r"\1", s)
    s = re.sub(r"([(\[$]) ", r"\1", s)
    s = re.sub(r" (n't|'s|'re|'ve|'m|'ll|'d)\b", r"\1", s)
    return re.sub(r'" (.*?) "', r'"\1"', s)


def contexts(lines, n, ctx=128):
    ids = tok.encode(" ".join(detok(s) for s in lines))
    x = torch.tensor(ids[: n * (ctx - 1)]).view(n, ctx - 1)
    return torch.cat([torch.full((n, 1), BOS), x], dim=1)


ted = [l for split in ["tst2012", "tst2013"] for l in (IWSLT / f"{split}.en").read_text(encoding="utf-8").splitlines()]
text_idx = contexts(ted, 80)
print(text_idx.shape, "|", tok.decode(text_idx[0, 1:40]))

In [ ]:
# explore: thí nghiệm 1, điểm của 144 head trên dãy ngẫu nhiên lặp lại (20 dãy, half 50) và trên văn bản thật
L, H, HALF = model.config.n_layer, model.config.n_head, 50
rep = lens.repeated_tokens(model.config.vocab_size, HALF, 20, torch.Generator().manual_seed(0), prefix=BOS)
start = time.perf_counter()
_, rep_cache = hooks.run_with_cache(model, rep)
_, text_cache = hooks.run_with_cache(model, text_idx[:20])
scores = {
    "induction": torch.stack([lens.induction_score(rep_cache[f"pattern.{l}"], HALF) for l in range(L)]),
    "duplicate": torch.stack([lens.duplicate_token_score(rep_cache[f"pattern.{l}"], HALF) for l in range(L)]),
    "prev (văn bản)": torch.stack([lens.prev_token_score(text_cache[f"pattern.{l}"]) for l in range(L)]),
}
print(f"{time.perf_counter() - start:.0f} s")
for name, s in scores.items():
    v, i = s.flatten().topk(6)
    print(f"{name:15s}", "  ".join(f"{j // H}.{j % H}={x:.2f}" for x, j in zip(v.tolist(), i.tolist())))

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (name, s) in zip(axes, scores.items()):
    im = ax.imshow(s.numpy(), vmin=0, vmax=1, cmap="Blues")
    ax.set_title(name); ax.set_xlabel("head"); ax.set_ylabel("tầng")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

In [ ]:
# explore: thí nghiệm 2, ablation: tắt k head induction mạnh nhất (zero và mean) so với tắt k head ngẫu nhiên không phải induction
rep2 = lens.repeated_tokens(model.config.vocab_size, HALF, 20, torch.Generator().manual_seed(1), prefix=BOS)
means = {l: text_cache[f"z.{l}"].mean(dim=(0, 1)) for l in range(L)}
order = scores["induction"].flatten().argsort(descending=True).tolist()
print(f"{'cấu hình':28s} loss bản 1   loss bản 2")
print(f"{'không ablation':28s} {in_context_loss(model, rep2, HALF)[0]:9.2f}   {in_context_loss(model, rep2, HALF)[1]:9.2f}")
for k in [1, 2, 4, 6]:
    heads = [(j // H, j % H) for j in order[:k]]
    for mode, m in [("zero", None), ("mean", means)]:
        with ablate_heads(model, heads, m):
            a, b = in_context_loss(model, rep2, HALF)
        print(f"{'top ' + str(k) + ' induction, ' + mode:28s} {a:9.2f}   {b:9.2f}")
others = [j for j in range(L * H) if scores["induction"].flatten()[j] < 0.1]
g = torch.Generator().manual_seed(2)
for k in [4, 6]:
    second = []
    for _ in range(5):
        pick = [others[p] for p in torch.randperm(len(others), generator=g)[:k].tolist()]
        with ablate_heads(model, [(j // H, j % H) for j in pick], means):
            second.append(in_context_loss(model, rep2, HALF)[1])
    print(f"{str(k) + ' head ngẫu nhiên, mean':28s} {'':9s}   " + ", ".join(f"{x:.2f}" for x in second))

In [ ]:
# explore: thí nghiệm 3, logit lens qua từng tầng trên 20 ngữ cảnh TED, và cùng phép đo khi quên ln_f
logits, cache = hooks.run_with_cache(model, text_idx[:20])
targets, final = text_idx[:20, 1:], logits[:, :-1]
print(f"{'stream':14s}    CE   top1=model    KL   hạng trung vị")
for name in ["resid_pre.0"] + [f"resid_post.{l}" for l in range(L)]:
    lg = lens.logit_lens(model, cache[name])[:, :-1]
    m = lens_metrics(lg, final, targets)
    rank = lens.token_rank(lg, targets).float().median().item()
    print(f"{name:14s} {m['ce']:6.2f}   {m['top1_agree']:8.3f}  {m['kl']:6.2f}   {rank:8.0f}")
with torch.no_grad():
    raw = model.lm_head(cache[f"resid_post.{L // 2}"])[:, :-1]
m = lens_metrics(raw, final, targets)
print(f"quên ln_f ở resid_post.{L // 2}: CE {m['ce']:.1f}, top1=model {m['top1_agree']:.3f}")

In [ ]:
# explore: thí nghiệm 4, probe "token đã xuất hiện chưa" qua từng tầng, với control task, baseline và GPT-2 khởi tạo ngẫu nhiên
y_all = repeated_token_labels(text_idx)[:, 1:].reshape(-1)
tok_all = text_idx[:, 1:].numpy().reshape(-1)
groups = np.repeat(np.arange(len(text_idx)), text_idx.shape[1] - 1)
tr, te = probe.group_split(groups, 0.25, 0)
y_ctrl = probe.control_labels(tok_all, y_all, 0)
print(f"{len(y_all)} token, tỉ lệ nhãn 1: {y_all.mean():.3f}, majority baseline: {probe.majority_baseline(y_all[tr], y_all[te]):.3f}")
seen = {}
for t, yv in zip(tok_all[tr], y_all[tr]):
    seen.setdefault(t, []).append(yv)
glob = int(round(y_all[tr].mean()))
lookup = np.array([int(round(np.mean(seen[t]))) if t in seen else glob for t in tok_all[te]])
print(f"baseline tra theo loại token (chỉ dùng token id): {(lookup == y_all[te]).mean():.3f}")

torch.manual_seed(0)
random_model = gpt_mini.GPT(model.config)
for block in random_model.blocks:
    block.mlp.act = torch.nn.GELU(approximate="tanh")
random_model.eval()
for label, mdl in [("GPT-2", model), ("khởi tạo ngẫu nhiên", random_model)]:
    _, c = hooks.run_with_cache(mdl, text_idx)
    print(f"\n{label}: stream          task  control  selectivity")
    for name in ["resid_pre.0"] + [f"resid_post.{l}" for l in range(L)]:
        X = c[name][:, 1:].reshape(-1, c[name].shape[-1]).numpy()
        Xtr, Xte = probe.standardize(X[tr], X[te])
        task = probe.probe_accuracy(probe.train_linear_probe(Xtr, y_all[tr], 2), Xte, y_all[te])
        ctrl = probe.probe_accuracy(probe.train_linear_probe(Xtr, y_ctrl[tr], 2), Xte, y_ctrl[te])
        print(f"{name:26s} {task:.3f}  {ctrl:.3f}   {task - ctrl:+.3f}")

In [ ]:
# explore: (tuỳ chọn) cùng các phép đo trên GPT mini tiếng Việt bạn train ở module 05 (đặt biến môi trường GPT_MINI_CKPT)
ckpt = os.environ.get("GPT_MINI_CKPT")
if not ckpt or not Path(ckpt).exists():
    print("Bỏ qua: chưa đặt GPT_MINI_CKPT tới checkpoint gpt-mini-v1 của module 05 (bài Advanced 01).")
else:
    from tokenizers import Tokenizer

    blob = torch.load(ckpt, map_location="cpu", weights_only=True)
    assert blob["format"] == "gpt-mini-v1"
    mini = gpt_mini.GPT(gpt_mini.GPTConfig(**blob["config"]))
    mini.load_state_dict(blob["state_dict"])
    mini.eval()
    vtok = Tokenizer.from_str(blob["tokenizer"])
    Lm, Hm = mini.config.n_layer, mini.config.n_head
    rep_vi = lens.repeated_tokens(mini.config.vocab_size, HALF, 20, torch.Generator().manual_seed(0))
    _, cm = hooks.run_with_cache(mini, rep_vi)
    ind = torch.stack([lens.induction_score(cm[f"pattern.{l}"], HALF) for l in range(Lm)])
    v, i = ind.flatten().topk(4)
    print("GPT mini, induction:", "  ".join(f"{j // Hm}.{j % Hm}={x:.2f}" for x, j in zip(v.tolist(), i.tolist())))
    vi_text = "\n".join(detok(s) for s in (IWSLT / "tst2013.vi").read_text(encoding="utf-8").splitlines())
    vids = torch.tensor(vtok.encode(vi_text).ids[: 20 * 128]).view(20, 128)
    lg_m, cm = hooks.run_with_cache(mini, vids)
    for name in [f"resid_post.{l}" for l in range(Lm)]:
        m = lens_metrics(lens.logit_lens(mini, cm[name])[:, :-1], lg_m[:, :-1], vids[:, 1:])
        print(f"GPT mini {name}: CE {m['ce']:.2f}, top1=model {m['top1_agree']:.3f}")

## 5. Đọc kết quả

Sau khi chạy hết, trả lời cho chính mình (rồi làm phần **trắc nghiệm** của bài trong app):

1. Bản đồ head: các head induction mạnh nằm ở những tầng nào, so với head duplicate và head previous-token? Thứ tự tầng đó có
   khớp với cơ chế hai bước của induction (mục 3 lý thuyết) không?
2. Ablation: so loss bản 2 khi tắt head induction với khi tắt cùng số head ngẫu nhiên. Zero và mean ablation khác nhau ở cột nào,
   và cột đó nói gì về việc zero ablation có "sạch" không?
3. Logit lens: từ tầng nào lens bắt đầu trùng top-1 với model ở đa số vị trí? Quên `ln_f` thì sao?
4. Probe: so accuracy ở `resid_pre.0` với baseline tra theo loại token, tầng tốt nhất với tầng cuối, GPT-2 với bản khởi tạo ngẫu
   nhiên, và cột selectivity. Kết quả cho phép nói gì, không cho phép nói gì về việc GPT-2 "dùng" thông tin này?